SPDX-FileCopyrightText:  PyPSA-Earth and PyPSA-Eur Authors
SPDX-License-Identifier: AGPL-3.0-or-later

# Validation of Zambia Model.
This notebook examines the outcomes generated by the PyPSA-Earth model in Zambia. Specifically, it compares publicly available data related to Zambia's power system with the information used and derived from the PyPSA-Earth model.

The following quantities will be reviewed:
 - inputs used by the pypsa-model:
    - network topology
    - installed capacity by type.
    - electricity demand
    - Interconnectors
 - outputs of the simulation:
   - electricity generation
   - power exports and imports
   - line flows per transmission line in zambia



## Import packages 

In [ ]:
import pypsa
import os
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.validation import make_valid
import pandas as pd
import numpy as np
import scipy as sp
import networkx as nx
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from cartopy.io.img_tiles import OSM
import cartopy.feature as cfeature
from scipy.sparse import csgraph
from itertools import product
import re

from shapely.geometry import Point, LineString
import shapely, shapely.prepared
from shapely.wkt import loads

## Preparation

In [ ]:
scenario_name = "validation_dispatch_zambia_2024"

scenario_subpath = scenario_name if scenario_name else ""
# OSM raw data files

osm_path = os.path.join("../", "resources", scenario_subpath, "osm")

raw_osm_path = os.path.join(osm_path, "raw")

clean_osm_path = os.path.join(osm_path, "clean")
assert os.path.exists(raw_osm_path), f"Raw OSM path does not exist: {raw_osm_path}"

substations_OSMraw_path = os.path.join(raw_osm_path, "all_raw_substations.geojson")
assert os.path.exists(
    substations_OSMraw_path
), f"Raw OSM path does not exist: {raw_osm_path}"

lines_OSMraw_path = os.path.join(raw_osm_path, "all_raw_lines.geojson")
assert os.path.exists(lines_OSMraw_path), f"Raw OSM path does not exist: {raw_osm_path}"


# cleaned osm data files
substations_OSMclean_path = os.path.join(
    clean_osm_path, "all_clean_substations.geojson"
)
assert os.path.exists(
    substations_OSMclean_path
), f"Clean OSM path does not exist: {clean_osm_path}"

lines_OSMclean_path = os.path.join(clean_osm_path, "all_clean_lines.geojson")

# shapes files
countries_shape_path = os.path.join(
    "../", "resources", scenario_subpath, "shapes", "country_shapes.geojson"
)
assert os.path.exists(
    countries_shape_path
), f"Country shapes path does not exist: {countries_shape_path}"

## Load Network

In [ ]:
network_path = os.path.join(
    "../", "results", scenario_subpath, "networks", "elec_s_10_ec_lv1.0_Co2L-3h.nc"
)
n = pypsa.Network(network_path)

## Validating Installed Capacities

In [ ]:
n.generators.query("carrier != 'load shedding' and carrier != 'export'").groupby(
    "carrier"
).p_nom.sum()

### Plotting

In [ ]:
carrier_capacity = (
    n.generators.query("carrier != 'load shedding' and carrier != 'export'")
    .groupby("carrier")
    .p_nom.sum()
)

carrier_capacity.plot.bar()
plt.ylabel("Installed Capacity (MW)")
plt.title("Installed Capacity by Carrier")
plt.show()

In [ ]:
n.storage_units.groupby("carrier").p_nom.sum()

### Plotting

In [ ]:
combined_capacity = carrier_capacity.add(
    n.storage_units.groupby("carrier").p_nom.sum(), fill_value=0
)
combined_capacity.plot.bar()
plt.ylabel("Installed Capacity (MW)")
plt.title("Installed Capacity by Carrier (including Storage)")
plt.show()

In [ ]:
erb_installed_capacities = {
    "hydro": 3024,
    "solar": 123,
    "coal": 330,
    "oil": 203,
    "biomass": 40,
    "ror": 138,
}

In [ ]:
erb_installed_capacities

In [ ]:
df_erb = pd.DataFrame.from_dict(
    erb_installed_capacities, orient="index", columns=["Installed Capacity (MW)"]
)
df_erb.plot.bar()
plt.ylabel("Installed Capacity (MW)")
plt.title("Installed Capacity by Carrier (ERB Data)")
plt.show()

### model installed capacities vs erb installed capacities

The source for the ERB data can be found here: https://www.erb.org.zm/wp-content/uploads/files/esr2024.pdf

In [ ]:
comparison_df = pd.DataFrame(
    {"Model": combined_capacity, "ERB": df_erb["Installed Capacity (MW)"]}
).fillna(0)

In [ ]:
plot_df = comparison_df.plot.bar()
plt.ylabel("Installed Capacity (MW)")
plt.title("Installed Capacity by Carrier: Model vs ERB Data")
plt.xticks(rotation=45)
plt.legend(title="Source")
plt.tight_layout()
plt.show()

## Electricity Demand 

In [ ]:
demand = (
    n.loads_t.p_set.multiply(n.snapshot_weightings.objective, axis=0).sum().sum() / 1e6
)

In [ ]:
print(f"{demand.round(2)} TWh")

According to the publicly available source 
[Charting the Globe – Zambia Electricity Demand](https://chartingtheglobe.com/region/zambia/energy/electricity-demand?indicator=demand&start=2020&end=2024&regions=248), 
Zambia’s electricity demand in **2023** was reported to be **16.6 TWh**.

However, the model estimates the electricity demand to be approximately **14 TWh**. 
The lower value produced by the model may be attributed to several factors, including the increasing adoption of **off-grid energy solutions**, which may not be fully captured in grid-based demand statistics. Additional differences may also arise from the assumptions used in the modelling framework, including demand aggregation, scaling methods, and temporal resolution.

## 1.3 Network Topology

### Analysis of the Zambian Transmission Network

In this specific section, a comparison is made regarding the publicly available data related to the network power system. The primary focus of this comparison lies in evaluating various aspects of the data, with special attention given to the main parameters being analyzed.

- Layout of the network as shown by images 
- Total length of the line

The data utilized for the comparison has been sourced from reliable public sources. 

In the Section in the first section, the network layout obtained by the workflow is drawn to be possibly compared with the image from the Zesco hosting capacity map.

In the Section 1.3.2, the network length as obtained from the workflow is calculated to then be compared with the information reported by [National Grid SLD 1_03_2022 Rev 8-Model (1)](https://drive.google.com/file/d/1UT5-A4BparyIv5Ye0olZep6f0yKL2-Rp/view?usp=sharing).





### 1.3.1 Network layout (using OSM clean data)
In this section, the cleaned OSM data is to draw plots of the entire network, to reproduce the images available from [Zesco hosting capacity map](https://www.zesco.co.zm/hosting_capacity.php#6/-13.253/28.004)  transmission network to verify the quality of the pypsa-earth model.

#### 1.3.1.1 Load clean OSM data

In [ ]:
# load substation geodataframe
df_substations_osm_clean = gpd.read_file(substations_OSMclean_path)
df_substations_osm_clean["geometry"] = df_substations_osm_clean["geometry"].apply(
    make_valid
)
# load lines geodataframe
df_lines_osm_clean = gpd.read_file(lines_OSMclean_path)
df_lines_osm_clean["geometry"] = df_lines_osm_clean["geometry"].apply(make_valid)

#### 1.3.3. Calculating the voltage levels for substations and lines
Substations voltage level [V]

In [ ]:
voltage_substations = df_substations_osm_clean.voltage.unique()
voltage_substations.sort()
voltage_substations

Lines voltage level [V]

In [ ]:
unique_voltages = df_lines_osm_clean.voltage.unique()
unique_voltages.sort()
unique_voltages

#### 1.3.1.2 Specifying colors for voltage levels 
Specify color array by voltage level

In [ ]:
color_voltages = ["orange", "blue", "red", "Purple", "brown", "green"]

voltage_to_color = {v: c for (v, c) in zip(unique_voltages, color_voltages)}
voltage_to_color

#### 1.3.1.3 Plotting the network data
Getting the borders of the country shape to perform the desired plot

In [ ]:
# get all country shapes
country_shapes = gpd.read_file(countries_shape_path)

# get zambia shape
df_geometry = country_shapes.set_index("name").geometry

# get bounds
# print("Original bounds:")
# print(df_geometry.boundary.bounds)

# add tolerance to bounds
tol = 0.2

bounds_mod = df_geometry.boundary.bounds
bounds_mod["minx"] -= tol  # reduce minx
bounds_mod["miny"] -= tol  # reduce miny
bounds_mod["maxx"] += tol  # increase maxx
bounds_mod["maxy"] += tol  # increase maxy

print("Modified bounds:")
print(bounds_mod)

# reorder bounds to comply with extend function (x0, x1, y0, y1)
extent_list = list(bounds_mod[["minx", "maxx", "miny", "maxy"]].loc["ZM"])

Drawing the picture using pyplot we get

In [ ]:
%matplotlib inline

# get the structure of the background data to plot
imagery = OSM()

max_width = 30  # max width of the figure
max_height = 30  # max heifht of the figure

# calculate figure size with appropriate multiplier to adhere to the desired width/height
multiplier = min(
    max_width / (extent_list[1] - extent_list[0]),
    max_height / (extent_list[3] - extent_list[2]),
)
figsize = (
    (extent_list[1] - extent_list[0]) * multiplier,
    (extent_list[3] - extent_list[2]) * multiplier,
)


fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(1, 1, 1, projection=imagery.crs)  # specify projection
ax.set_extent(extent_list, ccrs.PlateCarree())  # specify the location of the image
ax.add_image(imagery, 7)  # add the background image

# create an auxiliary dataframe for the substations with the desired crs and color properties
df_substations_osm_clean_plot = df_substations_osm_clean.to_crs(imagery.crs)
# specify the color of the nodes
df_substations_osm_clean_plot["color"] = df_substations_osm_clean_plot.voltage.apply(
    lambda x: voltage_to_color[x]
)

# create an auxiliary dataframe for the lines with the desired crs and properties
df_lines_osm_clean_plot = df_lines_osm_clean.to_crs(imagery.crs)
df_lines_osm_clean_plot["centroids"] = (
    df_lines_osm_clean_plot.geometry.boundary.centroid
)  # get the centroids of the line
df_lines_osm_clean_plot["color"] = df_lines_osm_clean_plot.voltage.apply(
    lambda x: voltage_to_color[x]
)  # specify the color of the line

# draw the substations
df_substations_osm_clean_plot.plot(color=df_substations_osm_clean_plot.color, ax=ax)

# draw the lines
df_lines_osm_clean_plot.plot(color=df_lines_osm_clean_plot.color, ax=ax)

# add annotations to show the number of circuits by line
for id, row in df_lines_osm_clean_plot.iterrows():
    ax.text(row.centroids.x, row.centroids.y, row.circuits, color=row.color)

In [ ]:
base_network_path = os.path.join("../", "networks", scenario_subpath, "elec.nc")
base_network = pypsa.Network(base_network_path)

Plot Base Network

In [ ]:
base_network.plot()

### 1.3.2 Calculate the total length of the lines
In this part, we calculate the overall length of the lines. 

#### 1.3.2.1 Calculate the total line length for OSM clean data
The information on total line length for the clean dataset are calculated.

Show line length by voltage level [km]

when multiple circuits are available, the length of the line is multiplied accordingly.

In [ ]:
# Note, since CRS EPSG:3763 is used, distances are in meters, thus by dividing for 1000, the units are in km
df_lines_osm_clean["length"] = (
    df_lines_osm_clean.to_crs(epsg=3857).geometry.length
    * df_lines_osm_clean.circuits
    / 1000
)
df_lines_osm_clean.groupby(by=["voltage"]).length.sum()

Total lines length [km]

In [ ]:
df_lines_osm_clean.to_crs(epsg=3857).length.sum() / 1000

#### 1.3.2.2 Remarks on the line length
Based on the available data from the Zambia Electricity Supply Corporation(ZESCO), zambia's network includes:

- 10,887.40 km of lines(2022) source


The current cleaning of OSM data captures approximately 114.7% (12487/10,887.40) of the network length. This value is quite close to the input data from OSM. The shape of the network in the OSM data closely resembles the image above. One factor affecting this could be  recent network upgrades might not be fully represented.

## 1.4.1 Interconnectors

In [ ]:
zam_links = n.links.query("bus0.str.contains('ZM') or bus1.str.contains('ZM')")

In [ ]:
for _, row in zam_links.iterrows():
    print(f"from {row['bus0']} to {row['bus1']}: {row['p_nom']} MW capacity")

## 2.1 Electricity Generation 

### 2.1.1 Electricity Generated By Power Plants

In [ ]:
valid_generators = n.generators.query(
    "carrier != 'load shedding' and carrier != 'export'"
).index

gen_energy_by_carrier_gwh = (
    n.generators_t.p.mul(n.snapshot_weightings.generators, axis=0)
    .groupby(n.generators.loc[valid_generators, "carrier"], axis=1)
    .sum()
    .sum()
    / 1e6
)

print(gen_energy_by_carrier_gwh)

In [ ]:
print(f"{gen_energy_by_carrier_gwh.sum().round(2)} TWh")

### 2.1.2 Electricity Generated By Reservoir Hydro Power Plants

In [ ]:
hydro_generation_twh = (
    n.storage_units_t.p.mul(n.snapshot_weightings.generators, axis=0).sum().sum() / 1e6
)
print(f"{hydro_generation_twh.round(2)} TWh")

Total Electricity Generated 

In [ ]:
print(
    f" Total Energy Generated: {float(hydro_generation_twh.round(2)) + float(gen_energy_by_carrier_gwh.sum().round(2))} TWh"
)

According to the Energy Regulation Board, the total energy generated in 2024 was 13.2 TWh, while the model estimates generation at about 11 TWh. The lower energy generation in the model could be due to an incorrect estimation of the impact of drought on hydropower plants. Hydro generation can be increased by adjusting the multiplier value in the configuration file.

### 2.2.3 Imports  and Exports

In [ ]:
flows_gwh = n.links_t.p0.mul(n.snapshot_weightings.objective, axis=0).sum() / 1e3

In [ ]:
imports = 0.0
exports = 0.0

for link in n.links.index:
    bus0 = n.links.loc[link, "bus0"]
    bus1 = n.links.loc[link, "bus1"]

    c0 = n.buses.loc[bus0, "country"]
    c1 = n.buses.loc[bus1, "country"]

    # only border links touching Zambia
    if c0 == "ZM":
        flow = flows_gwh[link]
        exports += max(flow, 0)
        imports += max(-flow, 0)

    elif c1 == "ZM":
        flow = flows_gwh[link]
        imports += max(flow, 0)
        exports += max(-flow, 0)

print("Imports (GWh):", imports)
print("Exports (GWh):", exports)

## 2.3 Zambia Transmission Line Flows

In [ ]:
total_line_flow_gwh = (
    n.lines_t.p0.abs().mul(n.snapshot_weightings.objective, axis=0).sum().sum() / 1000
)

print(total_line_flow_gwh, "GWh")